In [ ]:
!pip install datasets

In [ ]:
import torch
print(torch.cuda.get_device_name())

NVIDIA A100-SXM4-80GB


In [ ]:
# --- Colab setup -----------------------------------------------------------
from google.colab import files

import json, torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer

print("Upload TRAIN JSON:")
train_name = next(iter(files.upload()))
print("Upload TEST JSON:")
test_name = next(iter(files.upload()))

model_id = "google/t5-efficient-mini"
output_dir = "t5_efficient_mini_fp32"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_id)
label_pad_token_id = tokenizer.pad_token_id

def load_split(path):
    with open(path, "r", encoding="utf-8") as f:
        return Dataset.from_pandas(pd.DataFrame(json.load(f)["samples"]))

def preprocess(batch):
    model_inputs = tokenizer(
        batch["input"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["output"],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )["input_ids"]
    model_inputs["labels"] = [
        [(tok if tok != label_pad_token_id else -100) for tok in seq]
        for seq in labels
    ]
    return model_inputs

train_ds = load_split(train_name).map(preprocess, batched=True)
test_full = load_split(test_name)
split = test_full.train_test_split(test_size=0.5, seed=42)
val_ds = split["train"].map(preprocess, batched=True)
test_ds = split["test"].map(preprocess, batched=True)
dataset_test = test_ds  # alias for evaluation section

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
).to(device)

training_args = TrainingArguments(
    output_dir=output_dir,
    optim="adamw_torch",
    optim_args="foreach=False",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    num_train_epochs=30,
    bf16=False,
    fp16=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train()

Upload TRAIN JSON:


Saving dataset_1_operation_train.json to dataset_1_operation_train.json
Upload TEST JSON:


Saving dataset_1_operation_test.json to dataset_1_operation_test.json


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/125M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/125M [00:00<?, ?B/s]

Step,Training Loss
500,3.489400
1000,1.875700
1500,1.454800
2000,1.205800
2500,1.085300
3000,0.940300
3500,0.887200
4000,0.798300
4500,0.742300
5000,0.713000


TrainOutput(global_step=20340, training_loss=0.6709814981256146, metrics={'train_runtime': 970.8033, 'train_samples_per_second': 167.428, 'train_steps_per_second': 20.952, 'total_flos': 2357155516907520.0, 'train_loss': 0.6709814981256146, 'epoch': 30.0})

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

# 1. Resume training from the most recent checkpoint
output_dir = "t5_efficient_mini_fp32"
last_checkpoint = get_last_checkpoint(output_dir)
print(f"Resuming training from: {last_checkpoint}")

continued_args = TrainingArguments(
    output_dir=output_dir,
    optim="adamw_torch",
    optim_args="foreach=False",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    num_train_epochs=50,
    bf16=False,
    fp16=False,
    report_to=[],
)


trainer = Trainer(
    model=model,
    args=continued_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Resuming training from: t5_efficient_mini_fp32/checkpoint-20340


Step,Training Loss
20500,0.415900
21000,0.360300
21500,0.375900
22000,0.369700
22500,0.369000
23000,0.357300
23500,0.361400
24000,0.349500
24500,0.355700
25000,0.352800


TrainOutput(global_step=33900, training_loss=0.13626398429758077, metrics={'train_runtime': 653.5029, 'train_samples_per_second': 414.535, 'train_steps_per_second': 51.874, 'total_flos': 3928592528179200.0, 'train_loss': 0.13626398429758077, 'epoch': 50.0})

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


In [ ]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6db8f10a4652635003b10f62cd6e61186132ff37c8137e5d8ce3f4c038229a31
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import torch
import pandas as pd
import editdistance
from evaluate import load
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
dataset_test = test_ds
# --- helpers --------------------------------------------------------------
def levenshtein_similarity(preds, refs):
    scores = []
    for pred, ref in zip(preds, refs):
        pred = pred.strip()
        ref = ref.strip()
        dist = editdistance.eval(pred, ref)
        max_len = max(len(pred), len(ref))
        scores.append(1.0 - dist / max_len if max_len else 1.0)
    return sum(scores) / len(scores)

def bleu_score(preds, refs):
    smoothing = SmoothingFunction().method1
    vals = []
    for pred, ref in zip(preds, refs):
        pred_tokens = pred.strip().split()
        ref_tokens = ref.strip().split()
        if ref_tokens:
            vals.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing))
        else:
            vals.append(0.0)
    return sum(vals) / len(vals)

def rouge_scores(preds, refs):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1, rL = [], []
    for pred, ref in zip(preds, refs):
        scores = scorer.score(ref.strip(), pred.strip())
        r1.append(scores["rouge1"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)
    return {"rouge1": sum(r1) / len(r1), "rougeL": sum(rL) / len(rL)}

# --- prediction loop ------------------------------------------------------
model.eval()
predictions, references = [], []

for example in dataset_test:
    input_ids = torch.tensor(example["input_ids"], dtype=torch.long).unsqueeze(0).to(model.device)

    with torch.no_grad():
        generated = model.generate(input_ids, max_length=128)

    pred_text = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    predictions.append(pred_text)

    label_ids = example["labels"]
    if isinstance(label_ids, torch.Tensor):
        label_ids = label_ids.tolist()
    label_ids = [tok for tok in label_ids if tok != -100]
    ref_text = tokenizer.decode(label_ids, skip_special_tokens=True).strip()
    references.append(ref_text)

# --- metrics --------------------------------------------------------------
exact_match = load("exact_match").compute(predictions=predictions, references=references)["exact_match"]
lev_sim = levenshtein_similarity(predictions, references)
bleu = bleu_score(predictions, references)
rouge = rouge_scores(predictions, references)

results = pd.DataFrame([{
    "Model": "t5-efficient-mini-finetuned (bf32 training)",
    "Exact Match": exact_match,
    "Levenshtein Similarity": lev_sim,
    "BLEU Score": bleu,
    "ROUGE-1": rouge["rouge1"],
    "ROUGE-L": rouge["rougeL"],
}])

print(results)

                                         Model  Exact Match  \
0  t5-efficient-mini-finetuned (bf32 training)     0.651652   

   Levenshtein Similarity  BLEU Score   ROUGE-1   ROUGE-L  
0                0.890136    0.297478  0.878877  0.877493  


In [ ]:
import os
import shutil
# ---- Save compact FP32 model ----
fp32_dir = output_dir
trainer.save_model(fp32_dir)
tokenizer.save_pretrained(fp32_dir)

# Remove extra training artefacts (optimizer state, checkpoints)
for item in os.listdir(output_dir):
    if item.startswith("checkpoint") or item in {"trainer_state.json", "training_args.bin"}:
        path = os.path.join(output_dir, item)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

print(f"FP32 model saved to: {fp32_dir}")
print(f"Files: {os.listdir(fp32_dir)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

model_save_path_drive = "/content/drive/MyDrive/t5_efficient_mini_fp32(50)_2(65%)"

# Copy the model directory to Google Drive
shutil.copytree("t5_efficient_mini_fp32", model_save_path_drive, dirs_exist_ok=True)

print(f"Model saved to Google Drive at: {model_save_path_drive}")

In [1]:
!pip install optimum[onnxruntime]

In [4]:
import os
from google.colab import drive
from optimum.exporters.onnx import main_export

drive.mount('/content/drive')

# Google Drive path to your fine‑tuned model folder
MODEL_PATH = "/content/drive/MyDrive/t5_efficient_mini_fp32(50)_2(65%)"

# ONNX output directory (inside the same folder, like before)
ONNX_OUTPUT_DIR = os.path.join(MODEL_PATH, "onnx")

os.makedirs(ONNX_OUTPUT_DIR, exist_ok=True)

print(f"Exporting ONNX to: {ONNX_OUTPUT_DIR}")

# This mirrors your notebook:
main_export(
    model_name_or_path=MODEL_PATH,
    output=ONNX_OUTPUT_DIR,
    task="text2text-generation-with-past",
    opset=14,
    decoder=True,
    merge_decoder=True,
)

print("✓ ONNX export finished.")
print("ONNX files are now in:", ONNX_OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Exporting ONNX to: /content/drive/MyDrive/t5_efficient_mini_fp32(50)_2(65%)/onnx


Opset 14 is lower than the recommended minimum opset (18) to export t5. The ONNX export may fail or the exported model may be suboptimal.
/usr/local/lib/python3.12/dist-packages/transformers/models/t5/modeling_t5.py:1263: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py:108: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.keys is None or self.keys.numel() == 0:
Could not find ONNX initializer for torch parameter decoder.embed_tokens.weight. decoder.embed_tok

✓ ONNX export finished.
ONNX files are now in: /content/drive/MyDrive/t5_efficient_mini_fp32(50)_2(65%)/onnx
